# 05 — Heat Intelligence Report

`POST /v1/heat_intelligence` submits a multi-dimensional heat analysis. When the task finishes, the status endpoint streams a **PDF report** — the client writes it to `outputs/` for you.

**Plan:** Premium only.

Available analysis categories: `geographic`, `environmental`, `urban`, `events`, `anthropogenic`.

### Reference: the five analysis categories

Each value you pass in the `analysis=[...]` list adds a section to the generated PDF. You can request any subset — passing fewer keeps the report shorter and the credit cost lower. The keys (and what each section of the report actually covers, from the bundled sample):

| Key | Section in the PDF | What it contains |
|---|---|---|
| `geographic` | **Geographic Analysis** | General location, terrain & elevation, proximity to water, green spaces & vegetation, land cover, building density & height, shadow coverage, urban geometry & street-canyon effects |
| `environmental` | **Environmental Factors Analysis** | Air quality & atmospheric opacity, climate classification & historical weather, climate trends & projections, soil moisture & permeability, heat retention of surfaces, humidity, solar radiation, nighttime cooling, thermal comfort, seasonal UHI variability |
| `urban` | **Urban Factors Analysis** | Urban land use characteristics, dominant land use, impervious surface fraction, heat differentials from land use, zoning & planning, land use fragmentation & thermal equity, anthropogenic heat sources |
| `events` | **Events Analysis** | Extreme weather & heat event history, heatwave frequency & intensity, public health impacts & heat vulnerability |
| `anthropogenic` | **Anthropogenic Factors Analysis** | Heat emissions from vehicles & industry, transportation heat footprint, industrial / commercial waste heat, spatial heat concentration, temporal patterns at the requested date/time, emission-reduction measures, waste-heat recovery opportunities, cooling infrastructure & the AC feedback loop, balance-point temperature, cooling inequality, district cooling systems |

The bundled sample PDF (`data/real_estate_san_jose_heat_intelligence_sample_day_2024-10-02_p01.pdf`) was generated with all five categories enabled — open it for the full picture of what each section looks like on the page.

### Reference: how the response is delivered

Unlike the other endpoints, `heat_intelligence` does **not** return JSON. The status endpoint streams a finished **PDF report** directly. The client detects this and writes the file to disk:

- Default location: `outputs/heat_intelligence_<activity_id>.pdf`
- Override with `output_path=...` to write somewhere else.
- The call returns a `pathlib.Path` pointing at the saved file.

Because the response is a binary stream, `wait=False` is **not** supported on this method — the client always blocks until the PDF arrives.

### Reference: the `temperature` input

Like `environmental_parameters`, this endpoint requires a `temperature` value (ambient °C) to anchor the analysis. The bundled sample uses `temperature=40.74`, which appears in the PDF as the "baseline temperature" referenced throughout the Events and Anthropogenic sections. Source this from a heatmap call in real workflows — see [`notebooks/02_environmental_parameters.ipynb`](02_environmental_parameters.ipynb) for the same convention applied to the env-params endpoint.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv; load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
client = FortyGuardClient()

In [3]:
pdf_path = client.heat_intelligence(
    latitude=40.7128,
    longitude=-74.0060,
    temperature=32.5,
    date='2024-07-15',
    analysis=['environmental', 'urban'],
)
print(f'PDF saved to: {pdf_path.resolve()}')

Submitted -> activity_id=07261c0d-1dfc-4714-89ac-149de4d1a735
  status: processing
  status: processing
  status: processing


KeyboardInterrupt: 

In [ ]:
# Render an inline link to the generated report.
from IPython.display import FileLink
FileLink(str(pdf_path))